# Multilingual Customer Feedback Analyzer (Arabic + English)

## 1. Project Introduction

In this project we build a small NLP system that reads customer
reviews written in **Arabic** or **English** and predicts whether the
review is **Positive** or **Negative**.

The notebook does the following steps:

1. Load English and Arabic review datasets
2. Detect the language of every review
3. Clean and preprocess each language separately
4. Train three classical sentiment models on TF-IDF features
5. Evaluate them with Accuracy, Precision, Recall, F1-score
6. Compare Arabic vs English results
7. Extract the most important keywords using TF-IDF
8. Draw word clouds
9. Provide a function to test any new review by hand

We use classical Machine Learning (no deep learning).
Labels: **1 = Positive**, **0 = Negative**.


## 2. Import Libraries

If a library is missing, run the install line below. The cell also
downloads the small NLTK data files (stopwords + tokenizer).


In [ ]:
# Uncomment the next line the first time you run the notebook:
#!pip install pandas numpy scikit-learn nltk matplotlib seaborn wordcloud arabic-reshaper python-bidi joblib

import os
import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, ISRIStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)

import joblib

# Download NLTK data quietly
for pkg in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Could not download {pkg}: {e}")

print("All libraries imported successfully.")


## 3. Configuration and File Paths

All folders are relative to the notebook location, so this works on
Windows, macOS and Linux. The cell also creates any missing folders.


In [ ]:
BASE_DIR    = os.getcwd()
DATA_RAW    = os.path.join(BASE_DIR, "data", "raw")
DATA_PROC   = os.path.join(BASE_DIR, "data", "processed")
OUT_FIG     = os.path.join(BASE_DIR, "outputs", "figures")
OUT_MODELS  = os.path.join(BASE_DIR, "outputs", "models")
OUT_RESULTS = os.path.join(BASE_DIR, "outputs", "results")

for d in [DATA_RAW, DATA_PROC, OUT_FIG, OUT_MODELS, OUT_RESULTS]:
    os.makedirs(d, exist_ok=True)

# Expected dataset filenames (place them inside data/raw/)
ENGLISH_FILE = os.path.join(DATA_RAW, "Restaurant_Reviews.tsv")
ARABIC_FILE  = os.path.join(DATA_RAW, "balanced-reviews.csv")

RANDOM_STATE = 42

print("Folders are ready.")
print("English file expected at:", ENGLISH_FILE)
print("Arabic  file expected at:", ARABIC_FILE)


## 4. Load English Dataset

We use the **Kaggle Restaurant Reviews** dataset (TSV file).

- File name: `Restaurant_Reviews.tsv`
- Columns: `Review` (text), `Liked` (1 = positive, 0 = negative)

**Where to place it:** `data/raw/Restaurant_Reviews.tsv`.
If the file is missing, the cell prints clear instructions instead of
crashing.


In [ ]:
def load_english():
    if not os.path.exists(ENGLISH_FILE):
        print("English dataset NOT found.")
        print(f"Please download it and place it at:\n  {ENGLISH_FILE}")
        print("Kaggle search: 'Restaurant Reviews for Sentiment Analysis' (TSV).")
        return None
    df = pd.read_csv(ENGLISH_FILE, sep="\t", quoting=3)
    print("English dataset loaded:", df.shape)
    print(df.head())
    return df

df_en_raw = load_english()


## 5. Load Arabic Dataset

We use the **HARD (Hotel Arabic Reviews Dataset)** in CSV format.

- File name (example): `balanced-reviews.csv`
- Common columns: `review` (text), `rating` (1 to 5)

**Where to place it:** `data/raw/balanced-reviews.csv`.

Different distributions of the HARD dataset use slightly different
column names, so the code below tries to auto-detect them.


In [ ]:
import glob

def find_arabic_file():
    """Find the HARD file no matter its extension (.csv / .tsv / .txt)."""
    if os.path.exists(ARABIC_FILE):
        return ARABIC_FILE
    candidates = []
    for pattern in ["balanced-reviews.*", "*HARD*.*", "*hard*.*", "*arabic*review*.*"]:
        candidates.extend(glob.glob(os.path.join(DATA_RAW, pattern)))
    candidates = [c for c in candidates if os.path.splitext(c)[1].lower()
                  in {".csv", ".tsv", ".txt"}]
    return candidates[0] if candidates else None

def load_arabic():
    path = find_arabic_file()
    if path is None:
        print("Arabic dataset NOT found.")
        print(f"Please place the HARD dataset file inside:\n  {DATA_RAW}")
        print("Accepted file names: balanced-reviews.csv / .tsv / .txt")
        print("Reference: https://github.com/elnagara/HARD-Arabic-Dataset")
        return None

    # The HARD file is saved as UTF-16; other distributions use UTF-8 or
    # Windows-1256. Try each encoding until one works.
    encodings_to_try = ["utf-16", "utf-16-le", "utf-16-be",
                        "utf-8", "utf-8-sig", "cp1256"]
    last_error = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(
                path, sep=None, engine="python",
                encoding=enc, on_bad_lines="skip",
            )
            # Sanity check — the file must have at least 2 columns.
            if df.shape[1] < 2:
                last_error = f"Only {df.shape[1]} column(s) parsed with {enc}."
                continue
            print(f"Arabic dataset loaded from: {path}")
            print(f"Encoding used: {enc}")
            print("Shape:", df.shape)
            print("Columns:", list(df.columns))
            print(df.head())
            return df
        except (UnicodeDecodeError, UnicodeError) as e:
            last_error = e
            continue

    print("Could not read Arabic file with any tried encoding.")
    print("Last error:", last_error)
    return None

df_ar_raw = load_arabic()


## 6. Clean and Standardize Columns

Both datasets must end up with the same two columns:

- `text`  : the review content
- `label` : 1 = positive, 0 = negative

For the Arabic HARD dataset we convert star ratings to binary:

- ratings **4 or 5** → positive (1)
- ratings **1 or 2** → negative (0)
- rating **3 (neutral)** is dropped because this project is binary.


In [ ]:
def standardize_english(df):
    if df is None:
        return None
    df = df.rename(columns={"Review": "text", "Liked": "label"})
    df = df[["text", "label"]].dropna()
    df["label"] = df["label"].astype(int)
    return df.reset_index(drop=True)


def standardize_arabic(df):
    if df is None:
        return None
    cols = {c.lower(): c for c in df.columns}

    text_col = next((cols[c] for c in ["review", "text", "comment", "tweet", "content"] if c in cols), None)
    label_col = next((cols[c] for c in ["rating", "polarity", "label", "sentiment", "stars", "rate"] if c in cols), None)
    if text_col is None or label_col is None:
        print("Could not auto-detect Arabic columns. Found:", list(df.columns))
        return None

    df = df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"}).dropna()

    # Numeric labels (star ratings) -> binary, drop neutral 3
    if pd.api.types.is_numeric_dtype(df["label"]):
        df["label"] = pd.to_numeric(df["label"], errors="coerce")
        df = df.dropna(subset=["label"])
        if df["label"].max() > 1:
            df = df[df["label"] != 3]
            df["label"] = (df["label"] >= 4).astype(int)
        else:
            df["label"] = df["label"].astype(int)
    else:
        mapping = {"pos": 1, "positive": 1, "neg": 0, "negative": 0}
        df["label"] = df["label"].astype(str).str.lower().map(mapping)
        df = df.dropna(subset=["label"])
        df["label"] = df["label"].astype(int)

    return df.reset_index(drop=True)


df_en = standardize_english(df_en_raw)
df_ar = standardize_arabic(df_ar_raw)

if df_en is not None:
    print("English shape:", df_en.shape)
    print(df_en["label"].value_counts())
if df_ar is not None:
    print("Arabic shape:", df_ar.shape)
    print(df_ar["label"].value_counts())


## 7. Language Detection

For each review we add a `language` column. We use a simple, fast rule:
if the text contains any Arabic Unicode character (range
`\u0600`–`\u06FF`), label it `ar`; otherwise `en`. This is reliable
for our datasets and avoids adding extra dependencies like
`langdetect`.


In [ ]:
ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

def detect_language(text):
    if not isinstance(text, str):
        return "unknown"
    return "ar" if ARABIC_RE.search(text) else "en"

if df_en is not None:
    df_en["language"] = df_en["text"].apply(detect_language)
    print("Languages in English dataset:")
    print(df_en["language"].value_counts())

if df_ar is not None:
    df_ar["language"] = df_ar["text"].apply(detect_language)
    print("\nLanguages in Arabic dataset:")
    print(df_ar["language"].value_counts())


## 8. Language Detection Evaluation

Because every row already comes from a known-language dataset
(English file → `en`, Arabic file → `ar`), we get free ground-truth
labels for language detection. We compare our `detect_language` rule
against this ground truth, report accuracy, and print a few mistakes
if there are any.


In [ ]:
def evaluate_language_detection(df_en, df_ar):
    rows = []
    if df_en is not None:
        for txt, pred in zip(df_en["text"], df_en["language"]):
            rows.append({"true_lang": "en", "pred_lang": pred, "text": txt})
    if df_ar is not None:
        for txt, pred in zip(df_ar["text"], df_ar["language"]):
            rows.append({"true_lang": "ar", "pred_lang": pred, "text": txt})
    if not rows:
        print("No data to evaluate language detection.")
        return

    eval_df = pd.DataFrame(rows)
    eval_df["correct"] = eval_df["true_lang"] == eval_df["pred_lang"]
    n_total   = len(eval_df)
    n_correct = int(eval_df["correct"].sum())
    n_wrong   = n_total - n_correct
    acc       = n_correct / n_total

    print(f"Language detection accuracy: {acc:.4f}")
    print(f"Correct: {n_correct} / {n_total}")
    print(f"Wrong:   {n_wrong} / {n_total}")

    wrongs = eval_df[~eval_df["correct"]].head(5)
    if not wrongs.empty:
        print("\nExamples of wrong language detection (first 5):")
        for _, row in wrongs.iterrows():
            short = str(row["text"])[:100].replace("\n", " ")
            if len(str(row["text"])) > 100:
                short += "..."
            print(f"  true={row['true_lang']} pred={row['pred_lang']} | {short}")
    else:
        print("\nAll language predictions are correct.")

    summary = {"accuracy": acc, "correct": n_correct,
               "wrong": n_wrong, "total": n_total}
    path = os.path.join(OUT_RESULTS, "language_detection.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print("\nSaved:", path)

evaluate_language_detection(df_en, df_ar)


## 9. English Preprocessing

Steps:
1. Lowercase the text
2. Remove URLs and HTML tags
3. Remove punctuation and digits
4. Remove English stopwords (NLTK)
5. Apply Porter stemming


In [ ]:
try:
    EN_STOP = set(stopwords.words("english"))
except Exception:
    EN_STOP = set()

porter = PorterStemmer()

def preprocess_english(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [porter.stem(t) for t in tokens if t not in EN_STOP and len(t) > 1]
    return " ".join(tokens)

if df_en is not None:
    df_en["clean_text"] = df_en["text"].apply(preprocess_english)
    print(df_en[["text", "clean_text"]].head())


## 10. Arabic Preprocessing

Steps:
1. Remove diacritics (tashkeel)
2. Normalize alef forms (أ إ آ → ا)
3. Normalize ya (ى → ي) and ta marbuta (ة → ه)
4. Remove tatweel (ـ)
5. Keep only Arabic characters and whitespace
6. Remove Arabic stopwords (NLTK)
7. Apply the ISRI stemmer


In [ ]:
# Light Arabic preprocessing — NO stemming / no root extraction.
# Aggressive stemming (e.g. ISRIStemmer) was removing important letters
# from sentiment words like "ممتاز" → "متز". Keeping the words readable
# is more useful for sentiment analysis.

arabic_stopwords = set([
    "في", "من", "على", "الى", "إلى", "عن", "مع",
    "هذا", "هذه", "ذلك", "تلك",
    "كان", "كانت", "يكون",
    "هو", "هي", "هم", "هن", "انا", "أنا", "نحن",
    "و", "او", "أو", "ثم", "لكن",
    "لا", "ما", "لم", "لن", "قد",
    "جدا", "جداً", "كل", "أي", "اي",
])


def clean_arabic_text(text):
    if pd.isna(text):
        return ""
    text = str(text)

    # Remove Arabic diacritics
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    # Remove tatweel
    text = re.sub(r"ـ+", "", text)
    # Normalize Arabic letters
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    # Remove English letters and digits
    text = re.sub(r"[a-zA-Z0-9]", " ", text)
    # Keep Arabic letters and whitespace only
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    # Collapse extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Remove stopwords only — no stemming.
    words = [w for w in text.split() if w not in arabic_stopwords and len(w) > 1]
    return " ".join(words)


# Backwards-compatible alias so any other cell that still calls
# preprocess_arabic() keeps working.
preprocess_arabic = clean_arabic_text


if df_ar is not None:
    df_ar["clean_text"] = df_ar["text"].apply(clean_arabic_text)
    print("Before / after Arabic preprocessing (first 5 rows):")
    print(df_ar[["text", "clean_text"]].head().to_string(index=False))


> 📝 **Why light cleaning?** We used **light Arabic preprocessing**
> instead of aggressive stemming because Arabic root extraction can
> remove important letters and change the meaning of sentiment
> words. For example, *"ممتاز"* would become *"متز"* under root
> extraction, which destroys the sentiment signal. For sentiment
> analysis, keeping words readable is important.


## 11. Train / Validation / Test Split

Following the project proposal we split each language dataset into:

- **70 %** training set
- **15 %** validation set
- **15 %** test set

We use **stratified** sampling so the positive / negative ratio is
preserved in every split, and a fixed `random_state` so the results
can be reproduced.

> **Note on validation.** In this notebook we train a single fixed
> configuration per model, so we do **not** tune hyperparameters on
> the validation set yet. The validation split is kept aside for
> later use (grid search, threshold tuning, model selection). All
> scores reported in the evaluation section come from the **held-out
> test set** that the model never saw during training.


In [ ]:
def split_data(df, name):
    """Split into 70% train / 15% validation / 15% test (stratified)."""
    if df is None or df.empty:
        return None, None
    X = df["clean_text"].values
    y = df["label"].values
    # Step 1: peel off 30% as a temporary set (will become val + test).
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
    )
    # Step 2: split the 30% in half — 15% validation, 15% test.
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
    )
    print(f"[{name}] train: {len(X_train)}  val: {len(X_val)}  test: {len(X_test)}")
    # The train functions only need (X_train, X_test, y_train, y_test).
    # Validation is returned separately and held for future tuning.
    return (X_train, X_test, y_train, y_test), (X_val, y_val)


en_split, en_val = split_data(df_en, "English")
ar_split, ar_val = split_data(df_ar, "Arabic")

# Save the cleaned datasets for reference / reuse
if df_en is not None:
    df_en.to_csv(os.path.join(DATA_PROC, "english_processed.csv"), index=False)
if df_ar is not None:
    df_ar.to_csv(os.path.join(DATA_PROC, "arabic_processed.csv"), index=False)
print("Processed CSVs saved to:", DATA_PROC)


## 12. Baseline Model 1 — TF-IDF + Logistic Regression

TF-IDF turns text into numeric features that give more weight to rare
but informative words. Logistic Regression is a simple, strong baseline
for text classification.


In [ ]:
def train_lr(split, lang):
    if split is None:
        return None
    X_train, X_test, y_train, y_test = split
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2)
    Xtr = vec.fit_transform(X_train)
    Xte = vec.transform(X_test)
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    return {"vectorizer": vec, "model": model, "pred": pred, "y_test": y_test}


lr_en = train_lr(en_split, "english")
lr_ar = train_lr(ar_split, "arabic")

if lr_en:
    joblib.dump(lr_en["model"],      os.path.join(OUT_MODELS, "lr_english.joblib"))
    joblib.dump(lr_en["vectorizer"], os.path.join(OUT_MODELS, "tfidf_english.joblib"))
if lr_ar:
    joblib.dump(lr_ar["model"],      os.path.join(OUT_MODELS, "lr_arabic.joblib"))
    joblib.dump(lr_ar["vectorizer"], os.path.join(OUT_MODELS, "tfidf_arabic.joblib"))
print("Logistic Regression models saved to:", OUT_MODELS)


## 13. Baseline Model 2 — TF-IDF + Multinomial Naive Bayes

Naive Bayes is one of the simplest and fastest classifiers. It often
works surprisingly well on short reviews.


In [ ]:
def train_nb(split):
    if split is None:
        return None
    X_train, X_test, y_train, y_test = split
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2)
    Xtr = vec.fit_transform(X_train)
    Xte = vec.transform(X_test)
    model = MultinomialNB()
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    return {"vectorizer": vec, "model": model, "pred": pred, "y_test": y_test}


nb_en = train_nb(en_split)
nb_ar = train_nb(ar_split)

if nb_en: joblib.dump(nb_en["model"], os.path.join(OUT_MODELS, "nb_english.joblib"))
if nb_ar: joblib.dump(nb_ar["model"], os.path.join(OUT_MODELS, "nb_arabic.joblib"))
print("Naive Bayes models saved.")


## 14. Optional Model 3 — TF-IDF + Linear SVM

Linear SVMs are very strong on high-dimensional sparse features like
TF-IDF. We include them as a third comparison.


In [ ]:
def train_svm(split):
    if split is None:
        return None
    X_train, X_test, y_train, y_test = split
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2)
    Xtr = vec.fit_transform(X_train)
    Xte = vec.transform(X_test)
    model = LinearSVC(random_state=RANDOM_STATE)
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    return {"vectorizer": vec, "model": model, "pred": pred, "y_test": y_test}


svm_en = train_svm(en_split)
svm_ar = train_svm(ar_split)

if svm_en: joblib.dump(svm_en["model"], os.path.join(OUT_MODELS, "svm_english.joblib"))
if svm_ar: joblib.dump(svm_ar["model"], os.path.join(OUT_MODELS, "svm_arabic.joblib"))
print("SVM models saved.")


## 15. Evaluation Metrics

For every (language × model) pair we report:

- **Accuracy** — overall correct fraction
- **Precision** — of the predictions we called Positive, how many really were
- **Recall** — of the actual Positive reviews, how many we caught
- **F1-score** — harmonic mean of precision and recall
- **Classification report** — per-class precision / recall / F1, saved as a text file
- **Confusion matrix** — plotted in the next section

The summary table is saved to `outputs/results/all_metrics.csv`, and
one classification-report text file is saved per model under
`outputs/results/classification_report_<MODEL>.txt`.


In [ ]:
def evaluate(result, name):
    if result is None:
        return None
    y_test = result["y_test"]
    pred   = result["pred"]
    return {
        "model":     name,
        "accuracy":  accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, average="binary", zero_division=0),
        "recall":    recall_score(y_test, pred, average="binary", zero_division=0),
        "f1":        f1_score(y_test, pred, average="binary", zero_division=0),
    }


def save_classification_report(result, name):
    """Print and save a full classification report (per-class precision / recall / F1)."""
    if result is None:
        return
    report = classification_report(
        result["y_test"], result["pred"],
        target_names=["Negative", "Positive"],
        zero_division=0,
    )
    print(f"\n--- Classification report: {name} ---")
    print(report)
    path = os.path.join(OUT_RESULTS, f"classification_report_{name}.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"Classification report — {name}\n")
        f.write("=" * 40 + "\n\n")
        f.write(report)
    print("Saved:", path)


rows = []
for r, name in [
    (lr_en, "EN-LR"), (nb_en, "EN-NB"), (svm_en, "EN-SVM"),
    (lr_ar, "AR-LR"), (nb_ar, "AR-NB"), (svm_ar, "AR-SVM"),
]:
    m = evaluate(r, name)
    if m: rows.append(m)
    save_classification_report(r, name)

results_df = pd.DataFrame(rows)
print("\n=== Summary metrics ===")
print(results_df.round(4))

if not results_df.empty:
    out_csv = os.path.join(OUT_RESULTS, "all_metrics.csv")
    results_df.to_csv(out_csv, index=False)
    print("\nSaved:", out_csv)


## 16. Confusion Matrix

A confusion matrix shows the count of true positives, true negatives,
false positives, and false negatives. One plot is generated per
(language × model) pair, and saved into `outputs/figures/`.


In [ ]:
def plot_confusion(result, name):
    if result is None:
        return
    cm = confusion_matrix(result["y_test"], result["pred"])
    fig, ax = plt.subplots(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix — {name}")
    out_path = os.path.join(OUT_FIG, f"cm_{name}.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.show()
    print("Saved:", out_path)


for r, name in [
    (lr_en, "EN-LR"), (nb_en, "EN-NB"), (svm_en, "EN-SVM"),
    (lr_ar, "AR-LR"), (nb_ar, "AR-NB"), (svm_ar, "AR-SVM"),
]:
    plot_confusion(r, name)


## 17. Error Analysis

We look at the actual mistakes the best Logistic Regression model
made on the test set — **false positives** (predicted Positive but
really Negative) and **false negatives** (predicted Negative but
really Positive). Reading a few of these is the easiest way to
understand what the model is missing.

The examples are also saved to `outputs/results/errors_english.txt`
and `outputs/results/errors_arabic.txt`.


In [ ]:
def collect_errors(df, split, result, n_per_type=5):
    """Pull a few false positives and false negatives from the held-out test set."""
    if split is None or result is None or df is None:
        return [], []
    X_train, X_test, y_train, y_test = split
    pred = result["pred"]

    # Map cleaned text back to an original review for readability.
    text_lookup = df.groupby("clean_text")["text"].first().to_dict()

    fps, fns = [], []
    for clean, true_label, pred_label in zip(X_test, y_test, pred):
        original = text_lookup.get(clean, clean)
        if true_label == 0 and pred_label == 1 and len(fps) < n_per_type:
            fps.append(original)
        elif true_label == 1 and pred_label == 0 and len(fns) < n_per_type:
            fns.append(original)
        if len(fps) >= n_per_type and len(fns) >= n_per_type:
            break
    return fps, fns


def show_errors(lang_name, fps, fns, out_path=None):
    lines = [f"=== Error analysis — {lang_name} ==="]
    lines.append("\nFalse positives (predicted Positive, actually Negative):")
    if not fps:
        lines.append("  (none found in the first pass through the test set)")
    for original in fps:
        short = str(original)[:200].replace("\n", " ")
        if len(str(original)) > 200:
            short += "..."
        lines.append(f"  - {short}")
    lines.append("\nFalse negatives (predicted Negative, actually Positive):")
    if not fns:
        lines.append("  (none found in the first pass through the test set)")
    for original in fns:
        short = str(original)[:200].replace("\n", " ")
        if len(str(original)) > 200:
            short += "..."
        lines.append(f"  - {short}")
    output = "\n".join(lines)
    print(output)
    if out_path:
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(output)
        print("\nSaved:", out_path)


en_fps, en_fns = collect_errors(df_en, en_split, lr_en)
ar_fps, ar_fns = collect_errors(df_ar, ar_split, lr_ar)

show_errors("English (LR)", en_fps, en_fns,
            out_path=os.path.join(OUT_RESULTS, "errors_english.txt"))
print()
show_errors("Arabic (LR)",  ar_fps, ar_fns,
            out_path=os.path.join(OUT_RESULTS, "errors_arabic.txt"))


### Why the model gets these wrong

Looking at the examples above, the typical reasons our classical
TF-IDF model makes mistakes are:

1. **Sarcasm and irony.** *"Oh great, another cold meal."* contains
   positive words but means the opposite. Bag-of-words cannot catch
   this.
2. **Mixed reviews.** *"The food was good but the service was awful."*
   has both polarities. The model picks whichever class has stronger
   words on average.
3. **Negation.** *"not bad"* looks almost the same as *"bad"* to
   TF-IDF because word order is lost.
4. **Aggressive Arabic stemming.** The ISRI stemmer chops words down
   to 3-letter roots, so two different words can collapse to the same
   stem and confuse the model.
5. **Rare / out-of-vocabulary words.** Words filtered out by
   `min_df=2` carry no weight, so unusual but informative vocabulary
   is lost.

These are exactly the issues a deep multilingual model (e.g.
XLM-RoBERTa) would handle better — see the *Future Work* section.


## 18. Arabic vs English Comparison

We take the **best score** reached on each language across the three
models, then plot a side-by-side bar chart. This gives a fair
high-level comparison.


In [ ]:
if not results_df.empty:
    results_df["language"] = results_df["model"].str.startswith("AR").map({True: "Arabic", False: "English"})
    summary = results_df.groupby("language")[["accuracy", "precision", "recall", "f1"]].max()
    print("Best metrics per language:")
    print(summary.round(4))

    ax = summary.plot(kind="bar", figsize=(7, 4))
    ax.set_title("Best Scores: Arabic vs English")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=0)
    plt.tight_layout()
    out_path = os.path.join(OUT_FIG, "comparison_ar_vs_en.png")
    plt.savefig(out_path, dpi=120)
    plt.show()
    print("Saved:", out_path)

    summary.to_csv(os.path.join(OUT_RESULTS, "comparison_ar_vs_en.csv"))


## 19. Top Keywords using TF-IDF

For each language we compute the words with the **highest average
TF-IDF score** inside positive reviews and inside negative reviews.
These are the words that most characterize each class.


In [ ]:
def top_keywords(df, top_n=15):
    if df is None or df.empty:
        return None
    vec = TfidfVectorizer(max_features=5000, min_df=2)
    X = vec.fit_transform(df["clean_text"].fillna(""))
    feats = np.array(vec.get_feature_names_out())
    result = {}
    for label, label_name in [(1, "positive"), (0, "negative")]:
        mask = df["label"].values == label
        if mask.sum() == 0:
            continue
        scores = np.asarray(X[mask].mean(axis=0)).ravel()
        top_idx = scores.argsort()[::-1][:top_n]
        result[label_name] = list(zip(feats[top_idx], scores[top_idx]))
    return result


en_keys = top_keywords(df_en) if df_en is not None else None
ar_keys = top_keywords(df_ar) if df_ar is not None else None

def show_keys(keys, name):
    if not keys:
        return
    print(f"\nTop keywords ({name}):")
    for cls, items in keys.items():
        print(f"  {cls}:", [w for w, _ in items])

show_keys(en_keys, "English")
show_keys(ar_keys, "Arabic")

if en_keys:
    with open(os.path.join(OUT_RESULTS, "top_keywords_english.json"), "w", encoding="utf-8") as f:
        json.dump({k: [(w, float(s)) for w, s in v] for k, v in en_keys.items()},
                  f, ensure_ascii=False, indent=2)
if ar_keys:
    with open(os.path.join(OUT_RESULTS, "top_keywords_arabic.json"), "w", encoding="utf-8") as f:
        json.dump({k: [(w, float(s)) for w, s in v] for k, v in ar_keys.items()},
                  f, ensure_ascii=False, indent=2)


## 20. LDA Topic Modeling

To understand **what** people are actually talking about (not just
which words are positive or negative), we run **Latent Dirichlet
Allocation (LDA)** on each language. LDA is an unsupervised method
that groups words that often appear together — useful for spotting
themes like *food*, *service*, *location*, *price*, etc.

We fit a small **5-topic** model on each language and save the top
10 words per topic to text files in `outputs/results/`.


In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer


def run_lda(df, lang_name, n_topics=5, n_top_words=10):
    if df is None or df.empty:
        print(f"[{lang_name}] no data — skipping LDA")
        return None
    cv = CountVectorizer(max_features=2000, min_df=5, max_df=0.9)
    X = cv.fit_transform(df["clean_text"].fillna(""))
    if X.shape[1] == 0:
        print(f"[{lang_name}] not enough vocabulary for LDA")
        return None
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE,
        learning_method="batch",
        max_iter=20,
    )
    lda.fit(X)
    feats = np.array(cv.get_feature_names_out())
    topics = []
    for k, comp in enumerate(lda.components_):
        top_idx = comp.argsort()[::-1][:n_top_words]
        topics.append((k + 1, feats[top_idx].tolist()))
    return topics


def save_topics(topics, lang_name, out_name):
    if not topics:
        return
    path = os.path.join(OUT_RESULTS, out_name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"LDA topics — {lang_name}\n")
        f.write("=" * 40 + "\n\n")
        for k, words in topics:
            line = f"Topic {k}: " + ", ".join(words)
            print(line)
            f.write(line + "\n")
    print("Saved:", path)


print("=== LDA topics — English ===")
en_topics = run_lda(df_en, "English")
save_topics(en_topics, "English", "topics_english.txt")

print("\n=== LDA topics — Arabic ===")
ar_topics = run_lda(df_ar, "Arabic")
save_topics(ar_topics, "Arabic", "topics_arabic.txt")


## 21. Word Clouds (Optional)

We draw word clouds for positive and negative reviews. Arabic word
clouds need two helper libraries (`arabic-reshaper`, `python-bidi`) and
an Arabic-capable font. If anything is missing, the cell prints a
clear message and just skips that part.


In [ ]:
try:
    from wordcloud import WordCloud
    HAS_WC = True
except ImportError:
    HAS_WC = False

try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    HAS_AR_WC = True
except ImportError:
    HAS_AR_WC = False


def english_wordcloud(df):
    if not HAS_WC or df is None:
        return
    for label, name in [(1, "positive"), (0, "negative")]:
        text = " ".join(df[df["label"] == label]["clean_text"].fillna(""))
        if not text.strip():
            continue
        wc = WordCloud(width=600, height=400, background_color="white").generate(text)
        plt.figure(figsize=(7, 4))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"English — {name}")
        out_path = os.path.join(OUT_FIG, f"wordcloud_en_{name}.png")
        plt.savefig(out_path, dpi=120, bbox_inches="tight")
        plt.show()
        print("Saved:", out_path)


def arabic_wordcloud(df, font_path=None):
    if not HAS_WC or df is None:
        return
    if not HAS_AR_WC:
        print("arabic-reshaper / python-bidi not installed. Skipping Arabic word cloud.")
        return
    if font_path is None:
        # Look for an Arabic-capable font across Windows, macOS, and Linux.
        font_candidates = [
            r"C:\Windows\Fonts\arial.ttf",                      # Windows
            "/System/Library/Fonts/Supplemental/Arial.ttf",     # macOS
            "/Library/Fonts/Arial.ttf",                         # macOS
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",  # Linux
            "/usr/share/fonts/truetype/freefont/FreeSans.ttf",  # Linux
        ]
        font_path = next((p for p in font_candidates if os.path.exists(p)), None)
    if font_path is None:
        print("No Arabic-capable font found. Skipping Arabic word cloud. "
              "Pass font_path=... to render it.")
        return
    for label, name in [(1, "positive"), (0, "negative")]:
        text = " ".join(df[df["label"] == label]["clean_text"].fillna(""))
        if not text.strip():
            continue
        reshaped = arabic_reshaper.reshape(text)
        bidi_text = get_display(reshaped)
        wc = WordCloud(width=600, height=400, background_color="white",
                       font_path=font_path).generate(bidi_text)
        plt.figure(figsize=(7, 4))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"Arabic — {name}")
        out_path = os.path.join(OUT_FIG, f"wordcloud_ar_{name}.png")
        plt.savefig(out_path, dpi=120, bbox_inches="tight")
        plt.show()
        print("Saved:", out_path)


if HAS_WC:
    english_wordcloud(df_en)
    arabic_wordcloud(df_ar)
else:
    print("wordcloud library not installed. Install with: pip install wordcloud")


## 22. Live Demo Simulation

This is the final demo of the multilingual pipeline. Pass any review
(Arabic or English) to `predict_review(text)` and the function will:

1. **Detect** the language using our rule-based detector.
2. **Preprocess** the text using the matching language pipeline.
3. **Predict** the sentiment using the matching Logistic Regression
   model (our best performer).
4. **Return** the language, the cleaned text, and the predicted
   sentiment (Positive / Negative).

To prove the system works end-to-end, we test it on four labelled
reviews — one Positive and one Negative in each language.


In [ ]:
def predict_review(text):
    """Predict the sentiment of an Arabic or English review.

    Returns a dictionary:
        {
            "input": <original text>,
            "detected_language": "Arabic" | "English",
            "lang_code": "ar" | "en",
            "clean_text":  <preprocessed text>,
            "sentiment":   "Positive" | "Negative",
            "model":       <which model was used>,
            "confidence":  <float in [0, 1]>,
        }
    On failure, an "error" key is added with a friendly message.
    """
    lang = detect_language(text)

    # If the detector cannot decide, fall back to English.
    if lang not in ("ar", "en"):
        lang = "en"

    try:
        if lang == "ar":
            if lr_ar is None:
                return {
                    "input": text, "detected_language": "Arabic",
                    "lang_code": "ar", "language": "ar",
                    "sentiment": None, "model": None, "confidence": None,
                    "error": "Arabic model is not available.",
                }
            clean = clean_arabic_text(text)
            X = lr_ar["vectorizer"].transform([clean])
            proba = lr_ar["model"].predict_proba(X)[0]
            pred  = int(np.argmax(proba))
            conf  = float(proba[pred])
            model_name = "Arabic Logistic Regression"
        else:
            if lr_en is None:
                return {
                    "input": text, "detected_language": "English",
                    "lang_code": "en", "language": "en",
                    "sentiment": None, "model": None, "confidence": None,
                    "error": "English model is not available.",
                }
            clean = preprocess_english(text)
            X = lr_en["vectorizer"].transform([clean])
            proba = lr_en["model"].predict_proba(X)[0]
            pred  = int(np.argmax(proba))
            conf  = float(proba[pred])
            model_name = "English Logistic Regression"
    except Exception as e:
        return {
            "input": text, "detected_language": lang,
            "lang_code": lang, "language": lang,
            "sentiment": None, "model": None, "confidence": None,
            "error": f"Model error: {e}",
        }

    return {
        "input": text,
        "detected_language": "Arabic" if lang == "ar" else "English",
        "lang_code": lang,
        "language":  lang,       # backwards-compat alias
        "clean_text": clean,
        "sentiment":  "Positive" if pred == 1 else "Negative",
        "model":      model_name,
        "confidence": conf,
    }


# Live demo — one Positive and one Negative review in each language.
demo_reviews = [
    ("English (expected: Positive)",
     "The food was absolutely delicious and the service was great."),
    ("English (expected: Negative)",
     "Terrible experience. Cold food and rude staff."),
    ("Arabic  (expected: Positive)",
     "اخذت من عندهم عاملة أفريقية بنظام التأجير، العاملة جيدة وقابلة للتعلم. "
     "الاهم انهم الشركة متعاونين ويردوا على الاتصالات وتعاملهم للأمانة ممتاز."),
    ("Arabic  (expected: Negative)",
     "للأمانة التجربة كانت سيئة جدًا، لا يوجد أي مستوى من الشفافية في "
     "التعامل، وأسلوب الموظف غير مهني إطلاقًا."),
]

for label, review in demo_reviews:
    result = predict_review(review)
    print(f"[{label}]")
    print(f"  Review:     {review}")
    print(f"  Detected:   {result.get('detected_language')}")
    print(f"  Sentiment:  {result.get('sentiment')}")
    conf = result.get("confidence")
    if conf is not None:
        print(f"  Confidence: {conf:.4f}")
    print()


## 23. Live Demo: Interactive Multilingual Review Analyzer

This section gives an **interactive** demo right inside the notebook.
Type or paste a customer review in Arabic or English into the text
box below, then click **Analyze Review**. The system will:

1. **Detect** the language of the review.
2. **Preprocess** the text using the matching language pipeline.
3. **Predict** the sentiment using the trained Logistic Regression
   model.
4. **Display** the input, detected language, predicted sentiment,
   model used, and a confidence score.

You can also click an **example button** to quickly fill the text
box with a ready-made Arabic or English review, then click
**Analyze Review**.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output


# Pre-built example reviews (label, text).
EXAMPLES = [
    ("Arabic — Positive",  "المطعم ممتاز والأكل لذيذ والخدمة رائعة"),
    ("Arabic — Negative",  "التجربة سيئة جدا والأكل بارد والخدمة بطيئة"),
    ("English — Positive", "The food was amazing and the service was excellent"),
    ("English — Negative", "The restaurant was bad and the staff were rude"),
]


# --- Widgets ---
review_box = widgets.Textarea(
    value="",
    placeholder="Type or paste Arabic or English review here...",
    description="Review:",
    layout=widgets.Layout(width="100%", height="140px"),
)

analyze_btn = widgets.Button(
    description="Analyze Review",
    button_style="success",
    icon="check",
    tooltip="Run the trained model on the text above",
)

clear_btn = widgets.Button(
    description="Clear",
    button_style="warning",
    tooltip="Clear the text box and the output",
)

output = widgets.Output()


# --- Example fill buttons ---
def make_example_button(label, text):
    btn = widgets.Button(description=label,
                         layout=widgets.Layout(width="220px"))
    def fill(_):
        review_box.value = text
    btn.on_click(fill)
    return btn


example_row1 = widgets.HBox([
    make_example_button(EXAMPLES[0][0], EXAMPLES[0][1]),
    make_example_button(EXAMPLES[1][0], EXAMPLES[1][1]),
])
example_row2 = widgets.HBox([
    make_example_button(EXAMPLES[2][0], EXAMPLES[2][1]),
    make_example_button(EXAMPLES[3][0], EXAMPLES[3][1]),
])


# --- Button handlers ---
def on_analyze(_):
    with output:
        clear_output()
        text = review_box.value.strip()
        if not text:
            print("Please enter a review first.")
            return
        try:
            result = predict_review(text)
        except Exception as e:
            print("Sorry, something went wrong:", e)
            return
        if result.get("error"):
            print("Sorry, something went wrong:", result["error"])
            return
        conf = result.get("confidence")
        print("=== Live Demo Result ===")
        print(f"Input Review:        {result['input']}")
        print(f"Detected Language:   {result['detected_language']}")
        print(f"Predicted Sentiment: {result['sentiment']}")
        print(f"Model Used:          {result['model']}")
        if conf is not None:
            print(f"Confidence:          {conf:.4f}")


def on_clear(_):
    review_box.value = ""
    with output:
        clear_output()


analyze_btn.on_click(on_analyze)
clear_btn.on_click(on_clear)


# --- Display the demo ---
display(widgets.VBox([
    widgets.HTML("<b>Quick examples (click a button to fill the text box):</b>"),
    example_row1,
    example_row2,
    review_box,
    widgets.HBox([analyze_btn, clear_btn]),
    output,
]))


> 🎯 **For the final presentation:** run this cell, type or paste
> any Arabic or English review, then click **Analyze Review** to
> show the live prediction.


## 24. Conclusion, Limitations, and Future Work

### Conclusion
We built a complete multilingual pipeline that:

- loads English (Kaggle Restaurant Reviews) and Arabic (HARD) reviews,
- detects the review language with a simple Unicode-range rule
  (~99.94% accuracy on our datasets),
- cleans each language with its own pipeline (Porter stemming for
  English, **light cleaning without stemming** for Arabic),
- trains three classical models (Logistic Regression, Naive Bayes,
  Linear SVM) on TF-IDF features,
- evaluates them with Accuracy, Precision, Recall, F1, classification
  reports, and confusion matrices,
- extracts top TF-IDF keywords, runs LDA topic modelling, and draws
  word clouds,
- and exposes both a small `predict_review(text)` function and an
  **interactive ipywidgets live demo** at the end of the notebook.

The best Arabic model (Logistic Regression) reached
**F1 ≈ 93%** and the best English model reached **F1 ≈ 83%** on the
held-out test set.

### Limitations
- Classical TF-IDF features cannot capture context, word order, or
  sarcasm — *"not bad"* and *"bad"* look almost the same to the model.
- Aggressive Arabic stemming (ISRI) was tested earlier but removed
  important letters from sentiment words, so we switched to light
  cleaning.
- The HARD dataset is mostly **hotel** reviews while the English
  dataset is **restaurant** reviews, so the cross-language comparison
  is not perfectly fair.
- We only use binary sentiment (no Neutral class).
- The live demo depends on the trained models from this same notebook
  session — restart the kernel and run all cells before using it.

### Future Work
- Replace TF-IDF with multilingual transformer embeddings
  (e.g. `AraBERT`, `CAMeLBERT`, `xlm-roberta-base`,
  `bert-base-multilingual-cased`).
- Use **lemmatization** instead of stemming (e.g. spaCy for English,
  Farasa for Arabic).
- Add a Neutral class and turn the task into 3-way classification.
- Use the held-out 15% **validation split** for proper hyperparameter
  tuning.
- Wrap `predict_review` in a small **Streamlit** web app for use
  outside Jupyter.
